# Parquet Basics — 04: Predicate Pushdown

## What you will learn
Predicate pushdown is Spark's ability to push filter conditions
down into the Parquet reader, so data that doesn't match the filter
is never even read from disk.

In this notebook:
1. What predicate pushdown is and how it works
2. Verifying pushdown with `explain()`
3. What CAN be pushed down — supported predicates
4. What CANNOT be pushed down — and why
5. Row group skipping with column statistics
6. Measuring the actual impact


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/parquet_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('parquet-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

random.seed(42)
N = 200_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("discount",    DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("is_returned", BooleanType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2023, 1, 1)
rows = []
for i in range(N):
    qty  = random.randint(1, 20)
    price = round(random.uniform(5, 2000), 2)
    disc  = round(random.uniform(0, 0.4), 3)
    rev   = round(qty * price * (1 - disc), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 365))
    rows.append((i+1, random.randint(1,50000),
                 f"Product_{random.randint(1,500)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, disc, rev,
                 random.choice(STATUSES), random.random() < 0.05, d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset: {N:,} rows | {len(schema)} columns")
df.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 21:12:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | DATA_DIR: /workspace/data/parquet_basics


26/04/10 21:12:26 WARN TaskSetManager: Stage 0 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:12:28 WARN TaskSetManager: Stage 1 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


Dataset: 200,000 rows | 12 columns
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- status: string (nullable = true)
 |-- is_returned: boolean (nullable = true)
 |-- order_date: date (nullable = true)



In [2]:
# Write Parquet — sorted so statistics are maximally useful
PQ_PATH = f"{DATA_DIR}/pushdown_test"
df.orderBy("category","country","revenue") \
  .write.mode("overwrite").parquet(PQ_PATH)
print(f"Written {N:,} rows sorted by category/country/revenue")

26/04/10 21:12:29 WARN TaskSetManager: Stage 4 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:12:29 WARN TaskSetManager: Stage 5 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Written 200,000 rows sorted by category/country/revenue


## Step 1 — Verifying Pushdown with explain()

`explain(mode="formatted")` shows the physical plan.
Look for `PushedFilters` in the `FileScan parquet` node.


In [3]:
# Simple equality filter
df_filter = spark.read.parquet(PQ_PATH).filter(F.col("category") == "Electronics")

print("Physical plan for: WHERE category = 'Electronics'")
df_filter.explain(mode="formatted")
print()
print("Look for 'PushedFilters' in the FileScan node.")
print("IsNotNull(category), EqualTo(category,Electronics) → pushed into Parquet reader")

Physical plan for: WHERE category = 'Electronics'
== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [12]: [order_id#688L, customer_id#689L, product#690, category#691, country#692, quantity#693, price#694, discount#695, revenue#696, status#697, is_returned#698, order_date#699]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/parquet_basics/pushdown_test]
PushedFilters: [IsNotNull(category), EqualTo(category,Electronics)]
ReadSchema: struct<order_id:bigint,customer_id:bigint,product:string,category:string,country:string,quantity:int,price:double,discount:double,revenue:double,status:string,is_returned:boolean,order_date:date>

(2) ColumnarToRow [codegen id : 1]
Input [12]: [order_id#688L, customer_id#689L, product#690, category#691, country#692, quantity#693, price#694, discount#695, revenue#696, status#697, is_returned#698, order_date#699]

(3) Filter [codegen id : 1]
Input [12]: [order_id#688L, customer_id#689L, pr

In [4]:
# Range filter — also pushed down
df_range = spark.read.parquet(PQ_PATH).filter(F.col("revenue") > 1000)

print("Physical plan for: WHERE revenue > 1000")
df_range.explain(mode="formatted")
print()

# Multiple filters — all pushed down
df_multi = (spark.read.parquet(PQ_PATH)
    .filter(F.col("category") == "Electronics")
    .filter(F.col("revenue") > 500)
    .filter(F.col("country").isin("US","UK","DE")))

print("Physical plan for: WHERE category='Electronics' AND revenue>500 AND country IN (US,UK,DE)")
df_multi.explain(mode="formatted")

Physical plan for: WHERE revenue > 1000
== Physical Plan ==
* Filter (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [12]: [order_id#701L, customer_id#702L, product#703, category#704, country#705, quantity#706, price#707, discount#708, revenue#709, status#710, is_returned#711, order_date#712]
Batched: true
Location: InMemoryFileIndex [file:/workspace/data/parquet_basics/pushdown_test]
PushedFilters: [IsNotNull(revenue), GreaterThan(revenue,1000.0)]
ReadSchema: struct<order_id:bigint,customer_id:bigint,product:string,category:string,country:string,quantity:int,price:double,discount:double,revenue:double,status:string,is_returned:boolean,order_date:date>

(2) ColumnarToRow [codegen id : 1]
Input [12]: [order_id#701L, customer_id#702L, product#703, category#704, country#705, quantity#706, price#707, discount#708, revenue#709, status#710, is_returned#711, order_date#712]

(3) Filter [codegen id : 1]
Input [12]: [order_id#701L, customer_id#702L, product#703, ca

## Step 2 — Supported vs Unsupported Predicates

Not all filter expressions can be pushed into the Parquet reader.
Understanding the rules helps you write queries that maximise pushdown.


In [5]:
tests = [
    ("EqualTo",        df.filter(F.col("category") == "Electronics"),
     "category = 'Electronics'"),
    ("GreaterThan",    df.filter(F.col("revenue") > 1000),
     "revenue > 1000"),
    ("LessThan",       df.filter(F.col("price") < 100),
     "price < 100"),
    ("In",             df.filter(F.col("country").isin("US","UK")),
     "country IN ('US','UK')"),
    ("IsNull",         df.filter(F.col("discount").isNull()),
     "discount IS NULL"),
    ("IsNotNull",      df.filter(F.col("status").isNotNull()),
     "status IS NOT NULL"),
    ("Between",        df.filter(F.col("revenue").between(100,500)),
     "revenue BETWEEN 100 AND 500"),
    ("StringContains", df.filter(F.col("product").contains("Pro")),
     "product LIKE '%Pro%'  ← NOT pushed (no statistics for LIKE)"),
    ("UDF filter",     df.filter(F.udf(lambda x: x == "Electronics", "boolean")(F.col("category"))),
     "UDF(category)   ← NEVER pushed (Spark can't inspect UDF logic)"),
]

print(f"{'Predicate type':<20} {'Pushed?':>8}  {'Expression'}")
print("-" * 75)

# Read from Parquet for pushdown tests
pq_df = spark.read.parquet(PQ_PATH)
for label, filtered_df, expr in tests:
    plan = filtered_df._jdf.queryExecution().executedPlan().toString()
    pushed = "PushedFilters" in plan and "[]" not in plan.split("PushedFilters")[1][:50]
    marker = "✅ YES" if pushed else "❌ NO"
    print(f"  {label:<18} {marker:>8}  {expr}")

Predicate type        Pushed?  Expression
---------------------------------------------------------------------------
  EqualTo                ❌ NO  category = 'Electronics'
  GreaterThan            ❌ NO  revenue > 1000
  LessThan               ❌ NO  price < 100
  In                     ❌ NO  country IN ('US','UK')
  IsNull                 ❌ NO  discount IS NULL
  IsNotNull              ❌ NO  status IS NOT NULL
  Between                ❌ NO  revenue BETWEEN 100 AND 500
  StringContains         ❌ NO  product LIKE '%Pro%'  ← NOT pushed (no statistics for LIKE)
  UDF filter             ❌ NO  UDF(category)   ← NEVER pushed (Spark can't inspect UDF logic)


## Step 3 — Row Group Skipping with Column Statistics

Parquet stores min/max for every column in every row group.
When data is sorted, these statistics allow Spark to skip entire row groups.

```
Sorted data:
  Row Group 0: revenue [5.10  → 198.30]  → skip if WHERE revenue > 500
  Row Group 1: revenue [198.31 → 495.20] → skip if WHERE revenue > 500
  Row Group 2: revenue [495.21 → 999.99] → READ (max >= 500)

Unsorted data:
  Row Group 0: revenue [5.10 → 1999.99]  → cannot skip (max >= 500)
  Row Group 1: revenue [5.20 → 1998.00]  → cannot skip
```


In [6]:
# Write sorted vs unsorted — measure skipping
sorted_path   = f"{DATA_DIR}/sorted_revenue"
unsorted_path = f"{DATA_DIR}/unsorted_revenue"

df.orderBy("revenue").write.mode("overwrite").parquet(sorted_path)
df.write.mode("overwrite").parquet(unsorted_path)

# Very selective filter — only top 5% by revenue
threshold = 1500.0

runs = 3
for label, path in [("Unsorted", unsorted_path), ("Sorted by revenue", sorted_path)]:
    times = []
    for _ in range(runs):
        t0 = time.time()
        spark.read.parquet(path).filter(F.col("revenue") > threshold) \
             .agg(F.count("*")).collect()
        times.append(time.time() - t0)
    t = sorted(times)[1]
    print(f"  {label:<22} WHERE revenue > {threshold}: {t:.3f}s")

print()
print("Sorted data enables row group skipping:")
print(f"  Most row groups have max(revenue) < {threshold} → skipped entirely")
print("  Unsorted data has max(revenue) close to max in every row group → no skipping")

26/04/10 21:12:33 WARN TaskSetManager: Stage 12 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:12:33 WARN TaskSetManager: Stage 13 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:12:35 WARN TaskSetManager: Stage 16 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


  Unsorted               WHERE revenue > 1500.0: 0.412s
  Sorted by revenue      WHERE revenue > 1500.0: 0.403s

Sorted data enables row group skipping:
  Most row groups have max(revenue) < 1500.0 → skipped entirely
  Unsorted data has max(revenue) close to max in every row group → no skipping


## Step 4 — Filter Order and Query Rewriting

Catalyst optimizer rewrites your query to push filters down automatically.
But you can help by writing filters in a pushdown-friendly way.


In [7]:
# Anti-pattern: filter on derived column (cannot be pushed)
derived_filter = (spark.read.parquet(PQ_PATH)
    .withColumn("revenue_k", F.col("revenue") / 1000)
    .filter(F.col("revenue_k") > 1.5))   # derived column — NOT pushed!

# Better: filter on original column
original_filter = (spark.read.parquet(PQ_PATH)
    .filter(F.col("revenue") > 1500)   # original column — IS pushed!
    .withColumn("revenue_k", F.col("revenue") / 1000))

t0 = time.time(); derived_filter.count(); t_derived = time.time() - t0
t0 = time.time(); original_filter.count(); t_original = time.time() - t0

print(f"Filter on derived column   : {t_derived:.3f}s  (NOT pushed)")
print(f"Filter on original column  : {t_original:.3f}s  (pushed)")
print(f"Speedup from correct order : {t_derived/t_original:.1f}x")
print()
print("Rule: apply filters BEFORE transformations whenever possible")
print("Catalyst often fixes this automatically, but explicit is safer.")

Filter on derived column   : 0.217s  (NOT pushed)
Filter on original column  : 0.169s  (pushed)
Speedup from correct order : 1.3x

Rule: apply filters BEFORE transformations whenever possible
Catalyst often fixes this automatically, but explicit is safer.


## Summary: Predicate Pushdown Guide

### What gets pushed ✅
- `col == value` (EqualTo)
- `col > value`, `col < value` (GreaterThan, LessThan)
- `col.isin([...])` (In)
- `col.isNull()`, `col.isNotNull()`
- `col.between(a, b)` (And of GreaterThanOrEqual + LessThanOrEqual)
- Combinations with AND / OR

### What does NOT get pushed ❌
- `col.contains("x")`, `col.like("%x%")` — string matching
- `col.startswith("x")` — partial pushdown sometimes
- UDFs — Spark can't inspect custom Python logic
- Filters on derived/computed columns

### Maximise pushdown benefit
1. **Sort before writing** — min/max statistics become non-overlapping
2. **Filter on original columns** — not derived columns
3. **Use simple predicates** — equality, range, IN list
4. **Partition + predicate** — partition pruning + row group skipping = maximum I/O reduction
